In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForTokenClassification

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [3]:
class QASpanDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids,
                             attention_mask=attention_mask)
        return outputs

In [4]:
model = QASpanDetector()
model = model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [5]:
import pickle

# with open('data/debug_preprocessed2.pkl', 'rb') as f:
with open('data/debug_edge_case.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [6]:
len(train_encodings['input_ids'])

754

In [7]:
len(train_encodings['input_ids'][0])

512

In [8]:
len(train_encodings['attention_mask'][2])

512

In [9]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

In [11]:
def get_batch_answer_length(answer_start, reg_vector):
    reg_vector_temp = []
    for index, start in enumerate(answer_start):
        if start >= 512:
            reg_vector_temp.append(torch.FloatTensor([0]))
        else:
            reg_vector_temp.append(reg_vector[index, int(start)])
    
    reg_vector_temp = torch.FloatTensor(reg_vector_temp).reshape(len(answer_start), 1)
    return reg_vector_temp

In [38]:
import torch.nn.functional as F

def get_loss(reg_pred, obj_pred, reg_target, obj_target, answer_start):
    # Get answer length by get the the value at the answer start index of predict and target vector
    # reg_pred = torch.gather(reg_pred, 1, answer_start).exp()
    # reg_target = torch.gather(reg_target, 1, answer_start)

    batch_size = answer_start.size(0)
    if int(answer_start) >= 512:
        reg_loss = torch.zeros((1, 1))
    else:
        reg_pred = reg_pred[:, int(answer_start)].exp()
        reg_target = reg_target[:, int(answer_start)]
        # reg_loss = F.mse_loss(reg_pred, reg_target)
        intersection = torch.minimum(reg_pred, reg_target)
        union = torch.maximum(reg_pred, reg_target)
        print("### IOU Debug ###")
        print(reg_pred)
        print(reg_target)
        iou = intersection / union
        print(intersection)
        print(union)
        print(iou)
        reg_loss = -iou.log().sum() / batch_size
        print("## END Debug")
        
    # reg_pred = get_batch_answer_length(answer_start, reg_pred).to(device)
    # print(reg_pred.requires_grad)
    # reg_pred = reg_pred.exp()
    # reg_target = get_batch_answer_length(answer_start, reg_target)
    # print("Answer Start: ", answer_start)
    # print("Reg Pred:", reg_pred)
    # print("Reg Target:", reg_target)

    
    # reg_loss = F.mse_loss(reg_pred, reg_target)
    # obj_loss = F.binary_cross_entropy_with_logits(obj_pred, obj_target, reduction="sum") / batch_size
    # obj loss will return 0 when there are no answer 
    obj_loss = F.cross_entropy(obj_pred, answer_start.reshape((batch_size)), reduction="sum") / batch_size
    print("Reg Loss Backprop:", reg_loss.requires_grad)
    print("Obj Loss Backprop:", obj_loss.requires_grad)

    return reg_loss, obj_loss

In [39]:
MODEL_MAX_LENGTH = 512

In [40]:
import numpy as np

def create_labels(encodings):
    encodings['answer_length'] = np.array(encodings['end_positions'])\
        - np.array(encodings['start_positions']) + 1

    labels = np.zeros((len(encodings['input_ids']), MODEL_MAX_LENGTH, 
        2)) # num_example * seq_length * 2

    for example_idx, start in enumerate(encodings['start_positions']):
        if start < MODEL_MAX_LENGTH: # if the answer is not truncated
            labels[example_idx, start, 0] = 1
            labels[example_idx, start, 1] = encodings['answer_length'][example_idx]

    encodings['labels'] = labels

    return encodings

In [41]:
from torch.optim import AdamW

torch.manual_seed(0)
epochs = 3
print_every = 20
optim = AdamW(model.parameters(), lr=5e-5)
for epoch in range(epochs):
    # Set model in train mode
    model.train()
    loss_of_epoch = 0

    print("############Train############")
    for batch_idx, batch in enumerate(train_loader):
        batch = create_labels(batch)
        sentence_length = batch['input_ids'].size(1)
        batch_size = batch['input_ids'].size(0)

        answer_start = batch['start_positions'].reshape(batch_size, 1).to(device)        
        attention_mask = batch['attention_mask']
        attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

        reg_target = batch['labels'][:, :, 1]
        obj_target = batch['labels'][:, :, 0]
        obj_target = torch.FloatTensor(obj_target).to(device)
        reg_target = torch.FloatTensor(reg_target).to(device)

        outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))

        # print(outputs['logits'])
        reg_predict = outputs['logits'][:, :, 1]
        obj_predict = outputs['logits'][:, :, 0]
        reg_predict = F.pad(reg_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0).to(device)
        obj_predict = F.pad(obj_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = obj_predict * attention_mask.to(device)
        obj_predict = obj_predict.to(device)
        
        reg_loss, obj_loss = get_loss(reg_predict, obj_predict, reg_target, obj_target, answer_start)
        print("Reg Loss:", reg_loss)
        print("Obj Loss:", obj_loss)
        loss = reg_loss + obj_loss
        print("Loss:", loss)
        print()
        # loss.backward()
        # optim.step()
        # loss_of_epoch += loss.item()
        # if (batch_idx + 1) % print_every == 0:
        #     print("Batch {:} / {:}".format(batch_idx + 1, len(train_loader)))
        #     print("Loss:", round(loss.item(), 1), "\n")
        if batch_idx == 1:
            break
    break

############Train############
Reg Loss Backprop: False
Obj Loss Backprop: False
Reg Loss: tensor([[0.]])
Obj Loss: tensor([[0.]])
Loss: tensor([[0.]])

### IOU Debug ###
tensor([1.5000], grad_fn=<ExpBackward0>)
tensor([4.])
tensor([1.5000], grad_fn=<MinimumBackward0>)
tensor([4.], grad_fn=<MaximumBackward0>)
tensor([0.3750], grad_fn=<DivBackward0>)
## END Debug
tensor([[67]])
Reg Loss Backprop: True
Obj Loss Backprop: True
Reg Loss: tensor(0.9808, grad_fn=<DivBackward0>)
Obj Loss: tensor(6.2282, grad_fn=<DivBackward0>)
Loss: tensor(7.2090, grad_fn=<AddBackward0>)



In [42]:
outputs["logits"].sigmoid()

tensor([[[0.4610, 0.4343],
         [0.3932, 0.3521],
         [0.6240, 0.6126],
         ...,
         [0.5523, 0.4719],
         [0.5805, 0.4528],
         [0.5820, 0.3852]]], grad_fn=<SigmoidBackward0>)

In [43]:
F.cross_entropy(torch.Tensor([[-4.5410e-01, -1.3012e-01]]), torch.Tensor([[0., 0.]]), reduction="sum") / 1

tensor(-0.)

In [44]:
debug = torch.Tensor([[-4.5410e-01, -1.3012e-01, -2.4410e-01, -2.4603e-01, -2.4877e-01,
         -1.8318e-01, -4.4952e-01, -1.2397e-01, -2.7519e-03,  2.6314e-01,
          4.6617e-02, -8.8884e-02, -3.0135e-01, -1.3973e-01,  4.6212e-01,
          1.7456e-01, -4.5257e-02,  9.6171e-02,  2.1880e-01, -4.0610e-01,
          4.3104e-02,  2.3629e-01, -1.3451e-01, -1.0864e-01,  2.0145e-01,
         -4.2722e-01, -3.0015e-01,  1.2264e-01, -4.4774e-01, -1.0572e-01,
          8.0423e-02,  6.8256e-03, -3.5205e-01,  2.5784e-01,  5.0264e-01,
         -5.7846e-01, -7.2878e-02, -2.0102e-01,  6.2523e-01,  2.0517e-04,
          2.2452e-02,  1.3791e-01,  4.5307e-02,  1.0488e-01,  6.0359e-01,
          4.3872e-01, -1.6822e-01, -5.5352e-01,  1.8418e-01, -2.6748e-01,
          1.2132e-01,  1.8034e-01, -3.7811e-01, -2.3263e-01,  3.4349e-03,
         -6.4068e-01, -6.2396e-01, -2.9133e-01,  3.9295e-01, -1.4616e-01,
          2.5139e-01,  1.9395e-01,  2.1340e-01,  3.8293e-01, -4.8725e-01,
          4.6627e-02,  5.7616e-01, -3.6460e-01, -3.1909e-01,  2.5615e-01,
          6.6664e-01,  1.0311e-01,  5.0220e-02,  2.0549e-01, -7.7907e-01,
         -2.2772e-01,  1.0028e-01,  5.1139e-02,  1.9989e-01, -5.0557e-01,
         -3.5564e-01, -1.9734e-01,  4.9823e-01,  3.4986e-02,  3.9094e-02,
          5.6179e-01,  1.3642e-01, -3.2245e-01, -6.8802e-01, -3.1816e-01,
          1.6706e-01,  4.5599e-01, -2.4292e-02, -4.2062e-02, -2.0463e-01,
          1.0178e-01,  3.5490e-01,  2.2351e-01, -7.3183e-01, -1.3894e-02,
          4.9447e-01, -2.5024e-02, -3.4957e-02, -2.5253e-01,  1.7983e-01,
          2.7420e-01, -1.6475e-01, -3.5665e-01, -1.6097e-01, -4.1990e-01,
         -1.2531e-01, -4.4603e-01, -3.3023e-01, -7.2518e-01, -3.1750e-01,
         -7.8439e-01,  1.5360e-01, -5.8461e-01, -1.0711e-01, -4.1115e-01,
         -2.0905e-01,  2.9795e-01, -5.6755e-01,  7.1146e-02, -3.2548e-01,
         -1.6084e-01, -4.9780e-02, -4.2711e-01,  2.0436e-01,  1.7516e-01,
          2.9585e-01, -4.4444e-01, -2.3094e-01,  3.8771e-01,  6.9480e-03,
         -2.0725e-01, -1.2541e-01, -3.7217e-01, -8.3213e-01,  7.3777e-02,
         -2.9792e-02, -6.0441e-01,  3.2002e-02, -4.9009e-01, -6.0745e-01,
          1.8484e-01,  2.9113e-01, -4.0365e-02, -1.5809e-01,  5.0009e-01,
          5.3490e-01, -2.1911e-01, -9.8851e-01, -7.9413e-03,  1.7811e-02,
         -5.4680e-01, -9.6197e-03,  3.0297e-01, -1.9733e-01, -7.3787e-01,
          1.4462e-01,  2.0929e-01, -1.8006e-01,  2.0787e-01,  1.8456e-01,
          9.1600e-02, -7.5683e-02, -2.6512e-01, -9.6871e-02,  1.8381e-01,
          8.3492e-02,  4.1588e-01,  3.0709e-01,  1.3055e-01, -1.1806e-01,
         -8.3743e-01, -3.7809e-01, -3.7442e-01, -1.1988e-01, -4.8458e-01,
          1.0061e-01,  3.5446e-03, -6.5111e-02, -1.3527e-02, -2.9133e-01,
          1.5561e-01, -2.9940e-01, -1.2734e-01,  3.4312e-01, -1.4952e-01,
          1.1507e-01,  2.3461e-01, -6.8222e-02,  5.0100e-02, -5.2823e-01,
         -1.1917e-02, -2.4725e-01,  3.7340e-02,  2.7218e-01, -1.9366e-01,
         -4.0051e-02, -1.8797e-01, -3.0729e-01, -6.0960e-01, -6.1533e-02,
          1.7265e-01, -2.0165e-01, -2.9828e-01, -1.2038e-01,  5.5500e-02,
          3.3506e-01,  2.7954e-01,  6.6091e-01,  2.1922e-01,  1.5676e-01,
         -4.5344e-02, -4.1303e-01,  5.6198e-02, -4.6293e-01, -1.1442e-01,
         -1.8641e-01, -3.5282e-01,  2.7407e-01, -5.0997e-02, -1.2800e-01,
          4.4344e-01,  1.0312e-01, -2.2887e-01, -4.6214e-02, -2.1860e-01,
         -4.7614e-01, -3.9885e-01, -5.0185e-01, -3.7966e-01, -4.0407e-01,
         -2.0352e-01, -2.2839e-01, -2.2723e-01,  3.2706e-01,  2.8291e-01,
          4.6708e-01,  7.4355e-02,  1.1844e-01, -6.4665e-03,  5.5887e-02,
          5.6402e-02, -3.1974e-02,  5.4318e-01, -3.0099e-01, -4.0704e-01,
         -2.5709e-01, -1.8900e-01, -2.0317e-01,  6.1655e-01,  4.1782e-01,
          1.8373e-01,  3.4867e-02, -1.3581e-02,  2.9333e-02,  4.2188e-02,
         -5.3608e-02,  2.2942e-01,  3.2132e-01,  7.7869e-01,  4.7462e-01,
         -3.4829e-01, -2.0906e-01, -8.1863e-02,  5.3015e-01, -2.3538e-01,
          5.3748e-01,  4.6589e-01,  2.4409e-01, -2.2057e-01,  4.1492e-01,
          3.6153e-01,  1.5188e-01, -1.7878e-01, -2.5092e-01, -8.9830e-02,
          1.4256e-01, -3.3419e-01,  5.2927e-02, -5.5322e-01,  7.3176e-02,
         -4.2609e-02,  2.0945e-01, -3.6697e-01, -1.0647e-01, -4.7900e-01,
         -2.5330e-01, -2.8224e-01, -2.7727e-01, -6.4405e-02, -6.5231e-02,
         -6.2801e-01, -1.9451e-01, -9.4516e-02, -2.4295e-01,  8.5287e-02,
          5.9881e-02,  3.4419e-01,  1.8972e-01,  4.2985e-02, -7.0469e-02,
         -1.7157e-02, -1.9690e-01,  2.1026e-02, -2.4949e-01, -4.5714e-01,
         -8.8404e-02,  1.3299e-01, -6.0643e-01,  6.1837e-02, -1.4110e-01,
          5.2125e-01, -1.6907e-01,  2.3144e-01, -6.8268e-02,  3.7601e-01,
         -1.6411e-01,  1.5555e-02, -7.6516e-02, -1.0719e-01,  2.5572e-01,
          2.8134e-01, -1.5907e-02,  2.9505e-01,  2.5092e-01, -7.6093e-02,
          1.5513e-01, -8.4060e-02, -2.9293e-01, -1.1465e-01, -3.3013e-01,
          8.0810e-04,  2.4859e-01,  4.6341e-01,  4.1430e-01,  7.0672e-02,
         -8.3591e-03, -1.9333e-01, -3.4194e-01, -3.9407e-02, -2.8086e-01,
         -1.5441e-01, -4.6926e-01, -3.1976e-03, -7.7219e-01,  2.0092e-01,
         -6.1206e-02, -1.6323e-01,  4.5809e-02, -4.2898e-01, -6.4498e-01,
         -3.7175e-01, -3.6437e-01,  1.2529e-01,  1.4342e-01, -3.9244e-04,
         -3.2500e-01, -1.2238e-02, -2.2190e-01,  4.4821e-01,  3.5463e-01,
         -9.0348e-02, -1.8376e-02,  1.6632e-03,  1.0657e-02,  2.0656e-01,
         -5.9317e-03,  2.2283e-01, -1.9353e-01,  1.0175e-01, -5.5846e-02,
          8.9528e-02,  7.6902e-02,  9.5948e-02,  6.7838e-02, -1.5711e-01,
         -4.2603e-01, -1.6956e-01,  1.6695e-01, -1.1466e-01,  3.7293e-02,
         -2.4904e-01,  6.6346e-02,  5.4408e-01,  6.5584e-02,  1.9698e-01,
         -2.9761e-01, -4.0918e-01,  1.2402e-01,  1.1085e-01, -7.8929e-02,
          2.3637e-01, -3.1853e-01,  3.4657e-01,  3.8710e-01,  2.4079e-01,
          2.8430e-01, -5.2999e-01,  5.3359e-01, -8.8392e-02, -1.4536e-01,
          7.3618e-01,  2.6368e-02,  2.7749e-01, -5.2337e-02, -1.5755e-01,
         -1.9148e-01,  2.4125e-01,  6.2820e-02, -4.4108e-02, -3.6422e-01,
         -1.2174e-02,  6.6470e-02, -3.0820e-01, -4.0726e-01, -1.3209e-01,
          2.6048e-01,  5.7124e-01, -3.1427e-01,  1.1851e-01, -2.5725e-01,
         -2.8152e-01, -2.5252e-01,  5.8140e-01,  9.1614e-03, -2.5803e-01,
         -1.1399e-01, -2.2580e-01,  9.4483e-04,  9.2300e-02, -3.6786e-01,
         -2.4983e-01, -2.4233e-01, -2.6318e-01, -3.6834e-02,  1.1001e-01,
         -1.5307e-01, -5.5933e-02, -4.6156e-01, -7.5786e-02,  1.8177e-01,
          9.1329e-02, -8.0471e-02, -2.6843e-01,  1.2603e-01,  6.2556e-01,
          2.3520e-01,  1.5882e-01, -1.6851e-01,  2.5459e-01, -6.6602e-02,
         -3.3396e-01, -5.3402e-02,  1.3047e-01,  1.4875e-02, -1.5469e-01,
          1.2834e-01,  2.5721e-01,  1.8242e-01,  5.5063e-01, -5.8320e-02,
          3.0060e-01, -3.3819e-02,  1.1390e-01, -2.8486e-01, -1.0459e-01,
         -2.9321e-02,  2.2555e-01,  2.6015e-01, -7.1223e-02,  9.0914e-02,
          4.3545e-01,  1.1931e-01,  1.4837e-01,  3.8499e-02,  2.8640e-02,
          1.2136e-01, -4.4248e-01,  2.2376e-01, -4.4289e-02,  5.7159e-01,
          2.9995e-01,  4.9486e-02,  4.2852e-02, -3.9246e-02, -8.7539e-03,
          5.6674e-01,  2.8098e-01,  1.7429e-01,  2.6471e-01,  6.8941e-01,
          5.5122e-01, -4.9992e-01, -1.1609e-01, -2.8163e-01, -1.4380e-01,
         -1.8917e-01,  3.3080e-01,  2.5614e-01,  6.3194e-01,  2.3232e-01,
          2.1957e-01,  1.4879e-01, -5.8705e-01, -3.7610e-01, -4.1347e-01,
         -5.5077e-01,  1.0838e-01]])

In [45]:
debug.shape

torch.Size([1, 512])

In [46]:
test = F.log_softmax(debug)
# test = test.reshape((512, 1))
print(test.size())
target = torch.LongTensor([511])
print(F.nll_loss(test, target))

torch.Size([1, 512])
tensor(6.1448)


d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\ipykernel_launcher.py:1: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  """Entry point for launching an IPython kernel.


In [47]:
F.cross_entropy(debug, target)

tensor(6.1448)